# Appendix

### Import Libraries

In [166]:
from datetime import datetime
import kagglehub
import math
from matplotlib import gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind, chi2_contingency
from textwrap import dedent
import warnings

### Load Dataset

In [167]:
#Read the dataset directly form the link
url = "https://www.cancerimagingarchive.net/wp-content/uploads/RADCURE_Clinical_v04_20241219.xlsx"
df = pd.read_excel(url)
df.head()

,patient_id,Age,Sex,ECOG PS,Smoking PY,Smoking Status,Ds Site,Subsite,T,N,...,Local,Date Local,Regional,Date Regional,Distant,Date Distant,2nd Ca,Date 2nd Ca,RADCURE-challenge,ContrastEnhanced
0,RADCURE-0005,62.6,Female,ECOG 0,50,Ex-smoker,Oropharynx,post wall,T4b,N2c,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,0
1,RADCURE-0006,87.3,Male,ECOG 2,25,Ex-smoker,Larynx,Glottis,T1b,N0,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,1
2,RADCURE-0007,49.9,Male,ECOG 1,15,Ex-smoker,Oropharynx,Tonsil,T3,N2b,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,1
3,RADCURE-0009,72.3,Male,ECOG 1,30,Ex-smoker,Unknown,NaN,T0,N2c,...,NaN,NaT,NaN,NaT,NaN,NaT,S (suspicious),2008-05-27,0,0
4,RADCURE-0010,59.7,Female,ECOG 0,0,Non-smoker,Oropharynx,Tonsillar Fossa,T4b,N0,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,0


In [168]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3346 entries, 0 to 3345
Data columns (total 34 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   patient_id         3346 non-null   object        
 1   Age                3346 non-null   float64       
 2   Sex                3346 non-null   object        
 3   ECOG PS            3345 non-null   object        
 4   Smoking PY         3341 non-null   object        
 5   Smoking Status     3346 non-null   object        
 6   Ds Site            3346 non-null   object        
 7   Subsite            2972 non-null   object        
 8   T                  3334 non-null   object        
 9   N                  3333 non-null   object        
 10  M                  3332 non-null   object        
 11  Stage              3319 non-null   object        
 12  Path               3346 non-null   object        
 13  HPV                1717 non-null   object        
 14  Tx Modal

### Dataset Clean-up

In [169]:
#Clean-up and standardize column names
def clean_columns(col):
    col = str(col).strip()
    col = col.replace(" ", "_")
    col = col.replace("-", "_")
    return col.lower()

df.columns = [clean_columns(c) for c in df.columns]

#Convert blank numeric columns to NaN
numeric_cols = [
    "age",
    "dose",
    "fx",
    "length_fu",
    "contrastenhanced",
    "smoking_py"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

#Trim leading and extra spaces in object type columns
for col in df.columns:
    if df[col].dtype == "object":
      df[col] = (
      df[col]
      .str.strip()
      .str.replace(r"\s+", " ", regex=True))

#Fix the data in two features, Tx Modality and 2nd Ca
df["chemo"] = df["chemo"].replace("none", "No")
df["2nd_ca"] = df["2nd_ca"].replace("S (suspicious)", "S")

#Detect missing values and convert them to NaN
def make_nan(x):
    if isinstance(x, str):
        if x.strip().lower() in {"", "na", "n/a", "null", "unknown"}:
            return np.nan
    return x

for col in df.columns:
    df[col] = df[col].map(make_nan)

#Convert blank dates to NaT
date_cols = [
    "rt_start",
    "last_fu",
    "date_of_death",
    "date_local",
    "date_regional",
    "date_distant",
    "date_2nd_ca"
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

#Change the data in the HPV column
df["hpv"] = df["hpv"].replace({
    "Yes, positive": "Positive",
    "Yes, Negative": "Negative",
})

#Change target column to binary
df["status"] = df["status"].map({
        "Dead": 1,
        "Alive": 0,
    })

#Remove death related variables to prevent leakage
df = df.drop(columns=["cause_of_death", "date_of_death"])

#Remove pre-defined train and test split and patient_id
df = df.drop(columns=["radcure_challenge", "patient_id"])

#Remove local, regional, distant, 2n_ca date features due to too many blanks
df = df.drop(columns=["date_local", "date_regional", "date_distant", "date_2nd_ca"])

#Fill in Unknown for missing data on categorical features
df["hpv"] = df["hpv"].fillna("Unknown")
df["subsite"] = df["subsite"].fillna("Unknown")
df["ds_site"] = df["ds_site"].fillna("Unknown")
df["stage"] = df["stage"].fillna("Unknown")
df["smoking_status"] = df["smoking_status"].fillna("Unknown")
df["ecog_ps"] = df["ecog_ps"].fillna("Unknown")
df["t"] = df["t"].fillna("Unknown")
df["n"] = df["n"].fillna("Unknown")
df["m"] = df["m"].fillna("Unknown")
df["local"] = df["local"].fillna("Unknown")
df["regional"] = df["regional"].fillna("Unknown")
df["distant"] = df["distant"].fillna("Unknown")
df["ecog_ps"] = df["ecog_ps"].fillna("Unknown")
df["2nd_ca"] = df["2nd_ca"].fillna("Unknown")

#Fill in median value for missing numerical variables
df["smoking_py"] = df["smoking_py"].fillna(df["smoking_py"].median())

#Sanity check
print("\nShape after cleaning:", df.shape)
print("\nMissing values (top 5):")
print(df.isna().sum().sort_values(ascending=False).head(5))


Shape after cleaning: (3346, 26)

Missing values (top 5):
age               0
sex               0
ecog_ps           0
smoking_py        0
smoking_status    0
dtype: int64


In [170]:
from pathlib import Path
output_csv = Path("RADCURE_Clinical_cleaned.csv")
output_xlsx = Path("RADCURE_Clinical_cleaned.xlsx")
df.to_csv(output_csv, index=False)
df.to_excel(output_xlsx, index=False)
print(f"\nSaved cleaned CSV to:  {output_csv.resolve()}")
print(f"Saved cleaned XLSX to: {output_xlsx.resolve()}")


Saved cleaned CSV to:  /content/RADCURE_Clinical_cleaned.csv
Saved cleaned XLSX to: /content/RADCURE_Clinical_cleaned.xlsx
